# 1 对数据集的分析
## 1.1 修改标签文件路径

In [ ]:
import os
import xml.etree.ElementTree as ET
from pathlib import Path

def batch_modify_xml_path(xml_folder, target_prefix="./GC10-DET/"):
    """
    批量修改xml标签文件中的path为指定相对路径
    
    Args:
        xml_folder: 存放所有xml label文件的文件夹路径
        target_prefix: 相对路径的前缀（不包含类别文件夹和文件名部分）
    """
    # 遍历指定文件夹下所有xml文件
    for xml_file in Path(xml_folder).glob("*.xml"):
        try:
            # 解析xml文件
            tree = ET.parse(xml_file)
            root = tree.getroot()
            
            # 1. 获取文件名（从<filename>标签提取，保证和图片名一致）
            filename_elem = root.find("filename")
            if filename_elem is None or not filename_elem.text:
                print(f"警告：{xml_file.name} 中未找到<filename>标签，跳过")
                continue
            filename = filename_elem.text
            
            # 2. 提取类别文件夹（从<filename>所在的上级文件夹，或从object的name中提取）
            # 方式1：从object的name中提取（如3_yueyawan提取3）
            object_elem = root.find("object")
            if object_elem is not None:
                name = object_elem.find("name").text
                category = name.split("_")[0]  # 提取3
            else:
                # 方式2：如果没有object，从xml文件所在文件夹提取
                category = xml_file.parent.name
            
            # 3. 构建新的相对路径
            new_path = f"{target_prefix}{category}/{filename}"
            
            # 4. 修改path标签内容
            path_elem = root.find("path")
            if path_elem is None:
                # 如果path标签不存在，创建并添加
                path_elem = ET.SubElement(root, "path")
            path_elem.text = new_path
            
            # 5. 保存修改后的xml文件（覆盖原文件，建议先备份）
            tree.write(xml_file, encoding="utf-8", xml_declaration=True)
            
            print(f"成功修改：{xml_file.name} -> 新路径：{new_path}")
        
        except Exception as e:
            print(f"处理{xml_file.name}时出错：{str(e)}")

if __name__ == "__main__":
    # 配置：请修改为你的xml标签文件夹路径
    XML_FOLDER = r"./GC10-DET/label"  # 示例路径
    # 调用函数批量处理
    batch_modify_xml_path(XML_FOLDER)

## 1.2 标签与图片文件的分析
### 1.2.1 分析标签与图片是否一一对应，统计图片与标签的数量，并且统计每个缺陷的数量
各个缺陷标签的意义：

In [ ]:
类别名称        英文               标签
换卷冲孔 	    punching_hole	  1_chongkong
换卷焊缝        welding_line	  2_hanfeng
换卷月牙弯 	    crescent_gap	  3_yueyawan
斑迹-水斑	    water_spot	      4_shuiban
斑迹-油斑	    oil_spot	      5_youban
斑迹-丝斑	    silk_spot	      6_siban
异物压入 	    inclusion	      7_yiwu
压痕	       rolled_pit	     8_yahen
严重折痕	    crease	          9_zhehen
腰折	       waist folding	 10_yaozhe

In [18]:
import os
import xml.etree.ElementTree as ET
from collections import defaultdict

# 定义缺陷类别映射
DEFECT_CATEGORIES = {
    "1_chongkong": "换卷冲孔",
    "2_hanfeng": "换卷焊封",
    "3_yueyawan": "换卷月牙弯",
    "4_shuiban": "斑迹-水斑",
    "5_youban": "斑迹-油斑",
    "6_siban": "斑迹-丝斑",
    "7_yiwu": "异物压入",
    "8_yahen": "压痕",
    "9_zhehen": "严重折痕",
    "10_yaozhe": "腰折"
}

# 初始化统计变量
img_prefix_count = defaultdict(int)
defect_count = defaultdict(int)
# 存放图片缺失/解析异常的XML路径
missing_img_list = []

def parse_xml(xml_file_path):
    try:
        tree = ET.parse(xml_file_path)
        root = tree.getroot()
        
        # 检查path和图片是否存在
        path_elem = root.find('path')
        if path_elem is None:
            missing_img_list.append(xml_file_path)
            return
        
        img_path = path_elem.text.strip()
        if not os.path.exists(img_path):
            missing_img_list.append(xml_file_path)
        
        # 统计img_01~img_08
        filename_elem = root.find('filename')
        if filename_elem is not None:
            filename = filename_elem.text.strip()
            if filename.startswith("img_") and filename.endswith(".jpg"):
                prefix = filename.split('_')[0] + '_' + filename.split('_')[1]
                if prefix in [f"img_0{i}" for i in range(1,9)]:
                    img_prefix_count[prefix] += 1
        
        # 统计缺陷类别
        objects = root.findall('object')
        for obj in objects:
            name_elem = obj.find('name')
            if name_elem is None:
                continue
            defect_label = name_elem.text.strip()
            if defect_label in DEFECT_CATEGORIES:
                defect_count[defect_label] += 1
                
    except Exception:
        # 解析报错也计入错误
        missing_img_list.append(xml_file_path)

def batch_parse_xml(xml_dir):
    for file in os.listdir(xml_dir):
        if file.endswith(".xml"):
            xml_file = os.path.join(xml_dir, file)
            parse_xml(xml_file)

def save_error_to_txt():
    # 保存错误列表到txt
    with open("error_list.txt", "w", encoding="utf-8") as f:
        for path in missing_img_list:
            f.write(path + "\n")
    print(f"\n错误列表已保存至：error_list.txt")

# -------------------------- 执行 --------------------------
XML_DIR = "./GC10-DET/label/"  
batch_parse_xml(XML_DIR)

# 保存错误到文本
save_error_to_txt()

# 控制台输出错误信息
#print("【图片缺失/XML解析异常 文件列表】")
#for xml_path in missing_img_list:
    #print(xml_path)

print(f"错误文件总数：{len(missing_img_list)} 个")

# 图片前缀统计
print("\n【img_01 ~ img_08 图片数量统计】")
for prefix in sorted(img_prefix_count.keys()):
    print(f"   {prefix}: {img_prefix_count[prefix]} 张")
total_img = sum(img_prefix_count.values())
print(f"   总计（01~08）: {total_img} 张")

# 缺陷类别统计
print("\n【各缺陷类别数量统计】")
for label in sorted(DEFECT_CATEGORIES.keys()):
    cn_name = DEFECT_CATEGORIES[label]
    count = defect_count.get(label, 0)
    print(f"   {label}（{cn_name}）: {count} 个")
total_defect = sum(defect_count.values())
print(f"   缺陷总计: {total_defect} 个")


错误列表已保存至：error_list.txt
错误文件总数：0 个

【img_01 ~ img_08 图片数量统计】
   img_01: 364 张
   img_02: 371 张
   img_03: 387 张
   img_04: 167 张
   img_05: 145 张
   img_06: 332 张
   img_07: 318 张
   img_08: 210 张
   总计（01~08）: 2294 张

【各缺陷类别数量统计】
   10_yaozhe（腰折）: 143 个
   1_chongkong（换卷冲孔）: 329 个
   2_hanfeng（换卷焊封）: 513 个
   3_yueyawan（换卷月牙弯）: 265 个
   4_shuiban（斑迹-水斑）: 354 个
   5_youban（斑迹-油斑）: 569 个
   6_siban（斑迹-丝斑）: 884 个
   7_yiwu（异物压入）: 347 个
   8_yahen（压痕）: 85 个
   9_zhehen（严重折痕）: 74 个
   缺陷总计: 3563 个


### 1.2.2 修改错误标签
部分标签10_yaozhe被错误地标注为10_yaozhed，需要修改为10_yaozhe

In [7]:
import os
import xml.etree.ElementTree as ET

# ===================== 请修改这里为你的XML文件夹路径 =====================
XML_FOLDER = "./GC10-DET/label/"  # 你存放所有xml文件的根目录
# ======================================================================

# 错误标签 → 正确标签
ERROR_LABEL = "10_yaozhed"
CORRECT_LABEL = "10_yaozhe"

# 统计错误数量
fixed_count = 0
total_xml_count = 0

def fix_xml_label(xml_path):
    global fixed_count
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        need_save = False

        # 遍历所有缺陷目标
        objects = root.findall("object")
        for obj in objects:
            name_elem = obj.find("name")
            if name_elem is not None and name_elem.text == ERROR_LABEL:
                # 发现错误，进行修正
                name_elem.text = CORRECT_LABEL
                fixed_count += 1
                need_save = True

        # 如果有修改才保存
        if need_save:
            tree.write(xml_path, encoding="utf-8", xml_declaration=True)

    except Exception as e:
        print(f"处理失败 {xml_path}：{str(e)}")

def batch_fix_xml(folder_path):
    global total_xml_count
    # 遍历所有xml（包括子文件夹）
    for root_dir, _, files in os.walk(folder_path):
        for file in files:
            if file.endswith(".xml"):
                total_xml_count += 1
                xml_path = os.path.join(root_dir, file)
                fix_xml_label(xml_path)

if __name__ == "__main__":
    batch_fix_xml(XML_FOLDER)
    print(f"成功修正错误标签数量：{fixed_count} 个")


成功修正错误标签数量：131 个


### 1.2.3 修改图片文件路径错误
在1.1中，我们已经检查了图片文件路径是否存在，但是有些图片文件路径仍然是错误的。我们需要修改这些错误的图片文件路径，将其指向正确的图片文件。

In [ ]:
import os
import xml.etree.ElementTree as ET

# 配置
ERROR_TXT_PATH = "error_list.txt"

def fix_xml_path(xml_path):
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()

        # 读取folder和filename
        folder_val = root.find("folder").text.strip()
        file_name = root.find("filename").text.strip()

        # 构造标准正确path
        new_path = f"./GC10-DET/{folder_val}/{file_name}"

        # 修改path标签
        path_elem = root.find("path")
        if path_elem is None:
            path_elem = ET.SubElement(root, "path")
        path_elem.text = new_path

        # 保存，保持xml格式
        tree.write(xml_path, encoding="utf-8", xml_declaration=True)
        print(f"已修正：{xml_path} -> {new_path}")
        return True
    except Exception as e:
        print(f"修正失败：{xml_path}，错误：{e}")
        return False

def batch_fix_from_error_txt():
    # 读取错误列表
    with open(ERROR_TXT_PATH, "r", encoding="utf-8") as f:
        xml_lines = [line.strip() for line in f if line.strip()]

    success = 0
    fail = 0
    for xml_file in xml_lines:
        if os.path.exists(xml_file):
            if fix_xml_path(xml_file):
                success += 1
            else:
                fail += 1
        else:
            print(f"文件不存在：{xml_file}")
            fail += 1

    print("="*50)
    print(f"处理总数：{len(xml_lines)}")
    print(f"修正成功：{success}")
    print(f"修正失败：{fail}")

if __name__ == "__main__":
    batch_fix_from_error_txt()

# 2 特征工程
## 2.1 转换标注格式

In [1]:
import os
import xml.etree.ElementTree as ET

# 1. 类别映射（按你给的顺序）
class_list = [
    "1_chongkong",
    "2_hanfeng",
    "3_yueyawan",
    "4_shuiban",
    "5_youban",
    "6_siban",
    "7_yiwu",
    "8_yahen",
    "9_zhehen",
    "10_yaozhe"
]

# 2. 配置路径
xml_root_dir = "./GC10-DET/label"  # 你的XML标签文件根目录（会递归查找）
output_dir = "./GC10-DET/labels"      # 输出YOLO txt文件的目录

os.makedirs(output_dir, exist_ok=True)

def convert_xml_to_yolo(xml_path):
    """将单个XML文件转为YOLO格式txt"""
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()

        # 读取图片尺寸
        size_elem = root.find("size")
        img_w = int(size_elem.find("width").text)
        img_h = int(size_elem.find("height").text)

        # 生成txt文件名
        xml_name = os.path.basename(xml_path)
        txt_name = os.path.splitext(xml_name)[0] + ".txt"
        txt_path = os.path.join(output_dir, txt_name)

        with open(txt_path, "w", encoding="utf-8") as f:
            # 遍历所有目标
            for obj in root.findall("object"):
                name = obj.find("name").text.strip()
                # 处理你之前遇到的10_yaozhed错误
                if name == "10_yaozhed":
                    name = "10_yaozhe"

                # 映射类别id
                if name not in class_list:
                    print(f"警告：未知类别 {name}，跳过 -> {xml_path}")
                    continue
                class_id = class_list.index(name)

                # 读取bbox
                bbox = obj.find("bndbox")
                xmin = float(bbox.find("xmin").text)
                ymin = float(bbox.find("ymin").text)
                xmax = float(bbox.find("xmax").text)
                ymax = float(bbox.find("ymax").text)

                # VOC → YOLO 坐标转换
                # 归一化中心坐标和宽高
                x_center = ((xmin + xmax) / 2) / img_w
                y_center = ((ymin + ymax) / 2) / img_h
                w = (xmax - xmin) / img_w
                h = (ymax - ymin) / img_h

                # 写入一行
                f.write(f"{class_id} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}\n")

        return True
    except Exception as e:
        print(f"转换失败：{xml_path}，错误：{e}")
        return False

def batch_convert(xml_dir):
    """递归批量转换所有XML"""
    count = 0
    for root, _, files in os.walk(xml_dir):
        for file in files:
            if file.endswith(".xml"):
                xml_path = os.path.join(root, file)
                if convert_xml_to_yolo(xml_path):
                    count += 1
    print(f"\n转换完成！成功转换 {count} 个XML文件")
    print(f"输出目录：{os.path.abspath(output_dir)}")

if __name__ == "__main__":
    batch_convert(xml_root_dir)

警告：未知类别 d，跳过 -> ./GC10-DET/label\img_02_425616500_00770.xml

转换完成！成功转换 2294 个XML文件
输出目录：c:\Users\fengyarui1\Desktop\ReFormNet\GC10-DET\labels


## 2.2 统一图片路径

In [1]:
import os
import shutil

# ===================== 配置 =====================
# 根目录：1~10 文件夹所在的目录
SOURCE_ROOT = "./GC10-DET"
# 目标文件夹
TARGET_FOLDER = "./GC10-DET/images"
# =================================================

# 创建目标文件夹
os.makedirs(TARGET_FOLDER, exist_ok=True)

# 要遍历的文件夹 1~10
folders = [str(i) for i in range(1, 11)]

# 统计
moved_count = 0

for folder_name in folders:
    folder_path = os.path.join(SOURCE_ROOT, folder_name)
    
    if not os.path.exists(folder_path):
        print(f"跳过不存在的文件夹：{folder_path}")
        continue

    # 遍历当前文件夹里的所有文件
    for filename in os.listdir(folder_path):
        # 只移动图片格式（可自行增删）
        if filename.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
            src = os.path.join(folder_path, filename)
            dst = os.path.join(TARGET_FOLDER, filename)

            # 避免重名覆盖
            if os.path.exists(dst):
                print(f"已存在，跳过：{filename}")
                continue

            # 移动文件
            shutil.move(src, dst)
            moved_count += 1


print(f"总共移动图片数量：{moved_count} 张")
print(f"所有图片已存入：{os.path.abspath(TARGET_FOLDER)}")

已存在，跳过：img_03_425506300_00018.jpg
已存在，跳过：img_06_425502900_00052.jpg
已存在，跳过：img_07_425502900_00052.jpg
已存在，跳过：img_04_425503600_00017.jpg
已存在，跳过：img_07_425391800_00054.jpg
已存在，跳过：img_01_425503100_00018.jpg
已存在，跳过：img_07_425503000_00061.jpg
已存在，跳过：img_02_425392000_00984.jpg
已存在，跳过：img_03_436068500_00002.jpg
已存在，跳过：img_06_425505500_00052.jpg
已存在，跳过：img_06_3436814000_00687.jpg
已存在，跳过：img_07_436163600_01161.jpg
总共移动图片数量：2300 张
所有图片已存入：c:\Users\fengyarui1\Desktop\ReFormNet\GC10-DET\images


## 2.3 划分数据集

In [7]:
import os
import shutil
import random
from collections import defaultdict

# ===================== 配置 =====================
labels_dir = "./GC10-DET/labels"
images_dir = "./GC10-DET/images"
output_root = "./GC10-DET/yolo_dataset"
train_ratio = 0.8
val_ratio = 0.1
test_ratio = 0.1
random_seed = 42
# =================================================
random.seed(random_seed)

# 创建输出文件夹
for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(output_root, "images", split), exist_ok=True)
    os.makedirs(os.path.join(output_root, "labels", split), exist_ok=True)

# 1. 读取所有标签，以【文件第一个标签】作为文件所属类别
all_labels = [f for f in os.listdir(labels_dir) if f.endswith(".txt")]
# key:类别id  value:该类对应的所有文件名列表
class2files = defaultdict(list)
empty_files = []       # 无任何缺陷的空白文件
file_main_cls = dict() # 记录每个文件归属的主类别

print("正在解析标签，以文件第一个标签作为所属类别...")

for label_file in all_labels:
    txt_path = os.path.join(labels_dir, label_file)
    main_cls = None
    with open(txt_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            # 取第一个标签作为文件主类别
            cid = int(line.split()[0])
            main_cls = cid
            break
    if main_cls is not None:
        file_main_cls[label_file] = main_cls
        class2files[main_cls].append(label_file)
    else:
        empty_files.append(label_file)

# 2. 分层均匀划分：每一类内部单独按8:1:1切分
train_list = []
val_list = []
test_list = []

for cid, file_list in class2files.items():
    random.shuffle(file_list)
    total = len(file_list)
    train_num = int(total * train_ratio)
    val_num = int(total * val_ratio)
    
    train_list.extend(file_list[:train_num])
    val_list.extend(file_list[train_num : train_num+val_num])
    test_list.extend(file_list[train_num+val_num:])

# 空白无缺陷文件全部放入训练集
random.shuffle(empty_files)
train_list.extend(empty_files)

# 3. 统计划分后每一类在各集合中的数量
def count_cls_in_split(file_list):
    cnt = defaultdict(int)
    for f in file_list:
        if f in file_main_cls:
            cnt[file_main_cls[f]] += 1
    return cnt

train_cls_cnt = count_cls_in_split(train_list)
val_cls_cnt = count_cls_in_split(val_list)
test_cls_cnt = count_cls_in_split(test_list)

# 4. 输出统计结果
print("="*60)
print("各标签类别 在 训练集/验证集/测试集 数量分布")
print("="*60)
all_cids = sorted(class2files.keys())
for cid in all_cids:
    t = train_cls_cnt.get(cid, 0)
    v = val_cls_cnt.get(cid, 0)
    te = test_cls_cnt.get(cid, 0)
    total = t + v + te
    print(f"类别{cid:2d} | 训练:{t:2d}  验证:{v:2d}  测试:{te:2d}  总计:{total:2d}")

print("="*60)
print(f"总划分数量 -> 训练集:{len(train_list)}  验证集:{len(val_list)}  测试集:{len(test_list)}")
print(f"空白无缺陷文件数量：{len(empty_files)} (全部放入训练集)")
print("="*60)

# 5. 复制图片和标签到对应文件夹
def copy_files(file_list, split):
    for f in file_list:
        # 复制标签
        shutil.copy(os.path.join(labels_dir, f),
                    os.path.join(output_root, "labels", split, f))
        # 复制同名图片
        img_name = os.path.splitext(f)[0] + ".jpg"
        shutil.copy(os.path.join(images_dir, img_name),
                    os.path.join(output_root, "images", split, img_name))

print("正在复制文件...")
copy_files(train_list, "train")
copy_files(val_list, "val")
copy_files(test_list, "test")
print("数据集均匀划分完成！")

正在解析标签，以文件第一个标签作为所属类别...
各标签类别 在 训练集/验证集/测试集 数量分布
类别 0 | 训练:178  验证:22  测试:23  总计:223
类别 1 | 训练:229  验证:28  测试:30  总计:287
类别 2 | 训练:163  验证:20  测试:21  总计:204
类别 3 | 训练:233  验证:29  测试:30  总计:292
类别 4 | 训练:180  验证:22  测试:23  总计:225
类别 5 | 训练:524  验证:65  测试:66  总计:655
类别 6 | 训练:148  验证:18  测试:19  总计:185
类别 7 | 训练:24  验证: 3  测试: 4  总计:31
类别 8 | 训练:40  验证: 5  测试: 5  总计:50
类别 9 | 训练:112  验证:14  测试:14  总计:140
总划分数量 -> 训练集:1833  验证集:226  测试集:235
空白无缺陷文件数量：2 (全部放入训练集)
正在复制文件...
数据集均匀划分完成！


In [11]:
import os

# ===================== 配置 =====================
LABELS_FOLDER = "./GC10-DET/yolo_dataset/labels/val"  # 你的标签文件夹
TARGET_CLASS_ID = 7         # 你要查找的标签ID（例如 9 代表 10_yaozhe）
# =================================================

# 保存找到的文件
found_files = []

# 遍历所有标签
for txt_file in os.listdir(LABELS_FOLDER):
    if not txt_file.endswith(".txt"):
        continue
    
    txt_path = os.path.join(LABELS_FOLDER, txt_file)
    
    # 读取标签内容
    with open(txt_path, "r", encoding="utf-8") as f:
        lines = f.readlines()
    
    # 检查是否包含目标ID
    for line in lines:
        line = line.strip()
        if not line:
            continue
        class_id = line.split()[0]
        if class_id == str(TARGET_CLASS_ID):
            found_files.append(txt_file)
            break  # 找到就跳出，不重复记录

# ===================== 输出结果 =====================
print(f"🔍 查找类别 ID: {TARGET_CLASS_ID}")
print(f"✅ 找到文件数量: {len(found_files)} 个")
print("="*50)

# 打印文件名
for file in found_files:
    print(file)

# 保存到文件
with open(f"find_class_{TARGET_CLASS_ID}.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(found_files))

print(f"\n📄 结果已保存到: find_class_{TARGET_CLASS_ID}.txt")

🔍 查找类别 ID: 7
✅ 找到文件数量: 6 个
img_02_436068400_00001.txt
img_03_3403402800_00951.txt
img_03_425390000_00022.txt
img_03_4402117400_00003.txt
img_03_4402766900_00001.txt
img_03_4405964900_00010.txt

📄 结果已保存到: find_class_7.txt


# 3 模型训练
## 3.1 baseline

In [2]:
!pip install ultralytics -i https://pypi.tuna.tsinghua.edu.cn/simple

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     |████████████████████████████████| 1.3 MB 899 kB/s eta 0:00:01
     |████████████████████████████████| 32.5 MB 4.5 MB/s eta 0:00:011     |██████████████████              | 18.2 MB 650 kB/s eta 0:00:23


In [4]:
from ultralytics import YOLO
import torch

model = YOLO('yaml/baseline.yaml')

results = model.train(
    data='yolo_dataset/dataset.yaml',

    epochs=80,
    imgsz=1280,          # 从1280 → 960，大幅降低计算量（最有效）    
    
    batch=-1,           # 自动batch
    name='baseline',
    device=0,

    optimizer='SGD',
    lr0=0.01,
    momentum=0.937,
    weight_decay=0.0005,

    warmup_epochs=3,
    warmup_momentum=0.8,

    mosaic=0.0,       
    copy_paste=0.0,
    mixup=0.0,

    hsv_h=0.010,
    hsv_s=0.5,
    hsv_v=0.3,

    translate=0.05,     # 降低平移强度
    scale=0.3,          # 降低缩放强度
    fliplr=0.5,
    erasing=0.0,        # 关闭随机擦除（减少计算）

    amp=True,           # 保留混合精度（必须开）
    cache='ram',
    workers=8,          # 从16→8，降低CPU负载（防止卡顿）
    
    patience=50,
    save_period=10,

    nbs=64,             # 累积batch，稳定训练
    close_mosaic=0,     # 保持关闭mosaic
    dropout=0.0,        # 关闭dropout（小模型不需要）
    rect=False,         # 关闭矩形训练（减少预处理）
)

WARNING ⚠️ no model scale passed. Assuming scale='n'.
Ultralytics 8.4.54 🚀 Python-3.8.10 torch-2.0.0+cu118 CUDA:0 (NVIDIA GeForce RTX 4080 SUPER, 32229MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=0, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_dataset/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.0, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.01, hsv_s=0.5, hsv_v=0.3, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yaml/baseline.yaml, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=baseline, nbs=64, nms=False, op

In [5]:
model = YOLO('runs/detect/baseline/weights/best.pt')
# 评估模型
metrics = model.val(
    data='yolo_dataset/dataset.yaml',
    imgsz=1280
)

# 打印评估指标
print(f"mAP50: {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")

Ultralytics 8.4.54 🚀 Python-3.8.10 torch-2.0.0+cu118 CUDA:0 (NVIDIA GeForce RTX 4080 SUPER, 32229MiB)
baseline summary (fused): 101 layers, 1,880,541 parameters, 0 gradients, 4.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3626.8±358.8 MB/s, size: 377.3 KB)
val: Scanning /root/yolo_dataset/labels/val.cache... 204 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 204/204 35.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 3.0it/s 4.4s<0.1s
                   all        204        318      0.643      0.568      0.607      0.288
           1_chongkong         31         31      0.705          1      0.986      0.489
             2_hanfeng         50         50      0.107       0.26      0.201     0.0431
            3_yueyawan         26         26      0.892      0.956      0.939      0.546
             4_shuiban         31         34      0.859      0.588      0.679      0.379
              5_youb

## 3.2 baseline+M1
### 3.2.1 标签分为2*2

In [10]:
import os

# ===================== 配置 =====================
BOTTLENECK_CLASSES = {1,5,6}   # 瓶颈类别
GRID_SIZE = 8                      # n×n划分
OVERLAP_TH = 0.1                     # 交集比例阈值
EPS = 1e-6

# 原始标签路径
INPUT_TRAIN = "./yolo_dataset/labels/train_original"
INPUT_VAL   = "./yolo_dataset/labels/val_original"
INPUT_TEST  = "./yolo_dataset/labels/test_original"

# 输出路径
OUTPUT_TRAIN = "./yolo_dataset/labels/train"
OUTPUT_VAL   = "./yolo_dataset/labels/val"
OUTPUT_TEST  = "./yolo_dataset/labels/test"

os.makedirs(OUTPUT_TRAIN, exist_ok=True)
os.makedirs(OUTPUT_VAL, exist_ok=True)
os.makedirs(OUTPUT_TEST, exist_ok=True)


# ===================== 标签重构 =====================
def rga_reform_grid(txt_path, save_path):

    with open(txt_path,'r',encoding='utf-8') as f:
        lines=f.readlines()

    bboxes=[]

    for line in lines:

        line=line.strip()

        if not line:
            continue

        cls,x,y,w,h=map(float,line.split())

        bboxes.append(
            (int(cls),x,y,w,h)
        )

    new_lines=[]

    grid_w=1.0/GRID_SIZE
    grid_h=1.0/GRID_SIZE

    # 用来记录哪些目标被成功映射
    hit_index=set()

    # ===================== 遍历所有网格 =====================
    for i in range(GRID_SIZE):

        for j in range(GRID_SIZE):

            gx=grid_w*(j+0.5)
            gy=grid_h*(i+0.5)

            gw=grid_w
            gh=grid_h

            gx1=gx-gw/2
            gy1=gy-gh/2
            gx2=gx+gw/2
            gy2=gy+gh/2

            matched_cls=[]

            # ===================== 遍历所有缺陷 =====================
            for idx,(cls,x,y,w,h) in enumerate(bboxes):

                if cls not in BOTTLENECK_CLASSES:
                    continue

                bx1=x-w/2
                by1=y-h/2
                bx2=x+w/2
                by2=y+h/2

                # 计算交集
                ix1=max(gx1,bx1)
                iy1=max(gy1,by1)

                ix2=min(gx2,bx2)
                iy2=min(gy2,by2)

                iw=max(0,ix2-ix1+EPS)
                ih=max(0,iy2-iy1+EPS)

                inter=iw*ih

                box_area=w*h

                # 防止除0
                if box_area<EPS:
                    continue

                ratio=inter/box_area

                # 满足重叠阈值
                if ratio>=OVERLAP_TH:

                    matched_cls.append(cls)

                    hit_index.add(idx)

            # 去重
            matched_cls=list(set(matched_cls))

            # 保存多个类别
            for c in matched_cls:

                new_lines.append(
                    f"{c} "
                    f"{gx:.6f} "
                    f"{gy:.6f} "
                    f"{gw:.6f} "
                    f"{gh:.6f}"
                )

    # =====================
    # 未命中的瓶颈目标保留原标签
    # 防止标签彻底消失
    # =====================

    for idx,(cls,x,y,w,h) in enumerate(bboxes):

        if cls in BOTTLENECK_CLASSES:

            if idx not in hit_index:

                new_lines.append(
                    f"{cls} "
                    f"{x:.6f} "
                    f"{y:.6f} "
                    f"{w:.6f} "
                    f"{h:.6f}"
                )

        else:

            # 非瓶颈类原样保留
            new_lines.append(
                f"{cls} "
                f"{x:.6f} "
                f"{y:.6f} "
                f"{w:.6f} "
                f"{h:.6f}"
            )


    with open(save_path,'w',encoding='utf-8') as f:

        f.write('\n'.join(new_lines))


# ===================== 批量处理 =====================
def process_folder(input_dir,output_dir):

    files=os.listdir(input_dir)

    for fname in files:

        if fname.endswith(".txt"):

            rga_reform_grid(
                os.path.join(input_dir,fname),
                os.path.join(output_dir,fname)
            )


process_folder(INPUT_TRAIN,OUTPUT_TRAIN)
process_folder(INPUT_VAL,OUTPUT_VAL)
process_folder(INPUT_TEST,OUTPUT_TEST)


print("标签重构完成")
print("GRID:",GRID_SIZE)
print("OVERLAP_TH:",OVERLAP_TH)
print("瓶颈类别:",BOTTLENECK_CLASSES)

标签重构完成
GRID: 8
OVERLAP_TH: 0.1
瓶颈类别: {1, 5, 6}


In [9]:
from ultralytics import YOLO
import torch

model = YOLO('yaml/baseline.yaml')

results = model.train(
    data='yolo_dataset/dataset.yaml',

    epochs=80,
    imgsz=1280,          # 从1280 → 960，大幅降低计算量（最有效）    
    
    batch=-1,           # 自动batch
    name='baseline+M1',
    device=0,

    optimizer='SGD',
    lr0=0.01,
    momentum=0.937,
    weight_decay=0.0005,

    warmup_epochs=3,
    warmup_momentum=0.8,

    mosaic=0.0,       
    copy_paste=0.0,
    mixup=0.0,

    hsv_h=0.010,
    hsv_s=0.5,
    hsv_v=0.3,

    translate=0.05,     # 降低平移强度
    scale=0.3,          # 降低缩放强度
    fliplr=0.5,
    erasing=0.0,        # 关闭随机擦除（减少计算）

    amp=True,           # 保留混合精度（必须开）
    cache='ram',
    workers=8,          # 从16→8，降低CPU负载（防止卡顿）
    
    patience=50,
    save_period=10,

    nbs=64,             # 累积batch，稳定训练
    close_mosaic=0,     # 保持关闭mosaic
    dropout=0.0,        # 关闭dropout（小模型不需要）
    rect=False,         # 关闭矩形训练（减少预处理）
)

WARNING ⚠️ no model scale passed. Assuming scale='n'.
Ultralytics 8.4.54 🚀 Python-3.8.10 torch-2.0.0+cu118 CUDA:0 (NVIDIA GeForce RTX 4080 SUPER, 32229MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=0, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_dataset/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.0, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.01, hsv_s=0.5, hsv_v=0.3, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yaml/baseline.yaml, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=baseline+M1, nbs=64, nms=False,

In [10]:
model = YOLO('runs/detect/baseline+M1/weights/best.pt')
# 评估模型
metrics = model.val(
    data='yolo_dataset/dataset.yaml',
    imgsz=1280
)

# 打印评估指标
print(f"mAP50: {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")

Ultralytics 8.4.54 🚀 Python-3.8.10 torch-2.0.0+cu118 CUDA:0 (NVIDIA GeForce RTX 4080 SUPER, 32229MiB)
baseline summary (fused): 101 layers, 1,880,541 parameters, 0 gradients, 4.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 591.5±143.3 MB/s, size: 377.3 KB)
val: Scanning /root/yolo_dataset/labels/val.cache... 204 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 204/204 38.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 3.1it/s 4.1s<0.1s
                   all        204        439      0.479      0.509      0.518      0.205
           1_chongkong         31         31      0.513      0.935      0.898      0.437
             2_hanfeng         50         85      0.616      0.765      0.749      0.278
            3_yueyawan         26         26      0.403      0.546      0.471      0.146
             4_shuiban         31         34      0.307      0.353      0.209     0.0767
              5_youba

### 3.3 baseline+M1(16)

In [12]:
from ultralytics import YOLO
import torch

model = YOLO('yaml/baseline.yaml')

results = model.train(
    data='yolo_dataset/dataset.yaml',

    epochs=80,
    imgsz=1280,          # 从1280 → 960，大幅降低计算量（最有效）    
    
    batch=-1,           # 自动batch
    name='baseline+M1(16)',
    device=0,

    optimizer='SGD',
    lr0=0.01,
    momentum=0.937,
    weight_decay=0.0005,

    warmup_epochs=3,
    warmup_momentum=0.8,

    mosaic=0.0,       
    copy_paste=0.0,
    mixup=0.0,

    hsv_h=0.010,
    hsv_s=0.5,
    hsv_v=0.3,

    translate=0.05,     # 降低平移强度
    scale=0.3,          # 降低缩放强度
    fliplr=0.5,
    erasing=0.0,        # 关闭随机擦除（减少计算）

    amp=True,           # 保留混合精度（必须开）
    cache='ram',
    workers=8,          # 从16→8，降低CPU负载（防止卡顿）
    
    patience=50,
    save_period=10,

    nbs=64,             # 累积batch，稳定训练
    close_mosaic=0,     # 保持关闭mosaic
    dropout=0.0,        # 关闭dropout（小模型不需要）
    rect=False,         # 关闭矩形训练（减少预处理）
)

WARNING ⚠️ no model scale passed. Assuming scale='n'.
Ultralytics 8.4.54 🚀 Python-3.8.10 torch-2.0.0+cu118 CUDA:0 (NVIDIA GeForce RTX 4080 SUPER, 32229MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=0, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_dataset/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.0, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.01, hsv_s=0.5, hsv_v=0.3, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yaml/baseline.yaml, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=baseline+M1(16), nbs=64, nms=Fa

In [13]:
model = YOLO('runs/detect/baseline+M1(16)/weights/best.pt')
# 评估模型
metrics = model.val(
    data='yolo_dataset/dataset.yaml',
    imgsz=1280
)

# 打印评估指标
print(f"mAP50: {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")

Ultralytics 8.4.54 🚀 Python-3.8.10 torch-2.0.0+cu118 CUDA:0 (NVIDIA GeForce RTX 4080 SUPER, 32229MiB)
baseline summary (fused): 101 layers, 1,880,541 parameters, 0 gradients, 4.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2650.1±1312.4 MB/s, size: 377.3 KB)
val: Scanning /root/yolo_dataset/labels/val.cache... 204 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 204/204 38.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 2.4it/s 5.5s0.2s
                   all        204        676       0.68      0.329      0.362      0.169
           1_chongkong         31         31       0.69      0.935      0.886      0.426
             2_hanfeng         50        178          0          0     0.0333    0.00387
            3_yueyawan         26         26      0.753      0.808      0.786      0.392
             4_shuiban         31         34      0.785      0.265      0.451      0.237
              5_youb

## 3.4 baseline+M1(4)+M2

In [2]:
from ultralytics import YOLO
import torch

model = YOLO('yaml/baseline+M1+M2.yaml')

results = model.train(
    data='yolo_dataset/dataset.yaml',

    epochs=80,
    imgsz=1280,          # 从1280 → 960，大幅降低计算量（最有效）    
    
    batch=-1,           # 自动batch
    name='baseline+M1(4)+M2',
    device=0,

    optimizer='SGD',
    lr0=0.01,
    momentum=0.937,
    weight_decay=0.0005,

    warmup_epochs=3,
    warmup_momentum=0.8,

    mosaic=0.0,       
    copy_paste=0.0,
    mixup=0.0,

    hsv_h=0.010,
    hsv_s=0.5,
    hsv_v=0.3,

    translate=0.05,     # 降低平移强度
    scale=0.3,          # 降低缩放强度
    fliplr=0.5,
    erasing=0.0,        # 关闭随机擦除（减少计算）

    amp=True,           # 保留混合精度（必须开）
    cache='ram',
    workers=8,          # 从16→8，降低CPU负载（防止卡顿）
    
    patience=50,
    save_period=10,

    nbs=64,             # 累积batch，稳定训练
    close_mosaic=0,     # 保持关闭mosaic
    dropout=0.0,        # 关闭dropout（小模型不需要）
    rect=False,         # 关闭矩形训练（减少预处理）
)

WARNING ⚠️ no model scale passed. Assuming scale='n'.
New https://pypi.org/project/ultralytics/8.4.55 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.54 🚀 Python-3.8.10 torch-2.0.0+cu118 CUDA:0 (NVIDIA GeForce RTX 4080 SUPER, 32231MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=0, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_dataset/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.0, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.01, hsv_s=0.5, hsv_v=0.3, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=

## 3.5 baseline+M2

In [4]:
from ultralytics import YOLO
import torch

model = YOLO('yaml/baseline+M1+M2.yaml')

results = model.train(
    data='yolo_dataset/dataset.yaml',

    epochs=80,
    imgsz=1280,          # 从1280 → 960，大幅降低计算量（最有效）    
    
    batch=-1,           # 自动batch
    name='baseline+M2',
    device=0,

    optimizer='SGD',
    lr0=0.01,
    momentum=0.937,
    weight_decay=0.0005,

    warmup_epochs=3,
    warmup_momentum=0.8,

    mosaic=0.0,       
    copy_paste=0.0,
    mixup=0.0,

    hsv_h=0.010,
    hsv_s=0.5,
    hsv_v=0.3,

    translate=0.05,     # 降低平移强度
    scale=0.3,          # 降低缩放强度
    fliplr=0.5,
    erasing=0.0,        # 关闭随机擦除（减少计算）

    amp=True,           # 保留混合精度（必须开）
    cache='ram',
    workers=8,          # 从16→8，降低CPU负载（防止卡顿）
    
    patience=50,
    save_period=10,

    nbs=64,             # 累积batch，稳定训练
    close_mosaic=0,     # 保持关闭mosaic
    dropout=0.0,        # 关闭dropout（小模型不需要）
    rect=False,         # 关闭矩形训练（减少预处理）
)

WARNING ⚠️ no model scale passed. Assuming scale='n'.
New https://pypi.org/project/ultralytics/8.4.55 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.54 🚀 Python-3.8.10 torch-2.0.0+cu118 CUDA:0 (NVIDIA GeForce RTX 4080 SUPER, 32231MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=0, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_dataset/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.0, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.01, hsv_s=0.5, hsv_v=0.3, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=

## 3.6 baseline+M2+M3(conv)

In [4]:
from ultralytics import YOLO
import torch

model = YOLO('yaml/baseline+M1+M2+M3(conv).yaml')

results = model.train(
    data='yolo_dataset/dataset.yaml',

    epochs=80,
    imgsz=1280,          # 从1280 → 960，大幅降低计算量（最有效）    
    
    batch=-1,           # 自动batch
    name='baseline+M2+M3(conv)',
    device=0,

    optimizer='SGD',
    lr0=0.01,
    momentum=0.937,
    weight_decay=0.0005,

    warmup_epochs=3,
    warmup_momentum=0.8,

    mosaic=0.0,       
    copy_paste=0.0,
    mixup=0.0,

    hsv_h=0.010,
    hsv_s=0.5,
    hsv_v=0.3,

    translate=0.05,     # 降低平移强度
    scale=0.3,          # 降低缩放强度
    fliplr=0.5,
    erasing=0.0,        # 关闭随机擦除（减少计算）

    amp=True,           # 保留混合精度（必须开）
    cache='ram',
    workers=8,          # 从16→8，降低CPU负载（防止卡顿）
    
    patience=50,
    save_period=10,

    nbs=64,             # 累积batch，稳定训练
    close_mosaic=0,     # 保持关闭mosaic
    dropout=0.0,        # 关闭dropout（小模型不需要）
    rect=False,         # 关闭矩形训练（减少预处理）
)

WARNING ⚠️ no model scale passed. Assuming scale='n'.
New https://pypi.org/project/ultralytics/8.4.56 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.54 🚀 Python-3.8.10 torch-2.0.0+cu118 CUDA:0 (NVIDIA GeForce RTX 4080 SUPER, 32231MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=0, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_dataset/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.0, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.01, hsv_s=0.5, hsv_v=0.3, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=

## 3.6 baseline+M2+M3(conv)

In [9]:
from ultralytics import YOLO
import torch

model = YOLO('yaml/baseline+M1+M2+M3(scdown).yaml')

results = model.train(
    data='yolo_dataset/dataset.yaml',

    epochs=80,
    imgsz=1280,          # 从1280 → 960，大幅降低计算量（最有效）    
    
    batch=-1,           # 自动batch
    name='baseline+M2+M3(scdonw)',
    device=0,

    optimizer='SGD',
    lr0=0.01,
    momentum=0.937,
    weight_decay=0.0005,

    warmup_epochs=3,
    warmup_momentum=0.8,

    mosaic=0.0,       
    copy_paste=0.0,
    mixup=0.0,

    hsv_h=0.010,
    hsv_s=0.5,
    hsv_v=0.3,

    translate=0.05,     # 降低平移强度
    scale=0.3,          # 降低缩放强度
    fliplr=0.5,
    erasing=0.0,        # 关闭随机擦除（减少计算）

    amp=True,           # 保留混合精度（必须开）
    cache='ram',
    workers=8,          # 从16→8，降低CPU负载（防止卡顿）
    
    patience=50,
    save_period=10,

    nbs=64,             # 累积batch，稳定训练
    close_mosaic=0,     # 保持关闭mosaic
    dropout=0.0,        # 关闭dropout（小模型不需要）
    rect=False,         # 关闭矩形训练（减少预处理）
)

WARNING ⚠️ no model scale passed. Assuming scale='n'.


Traceback (most recent call last):
  File "/root/miniconda3/lib/python3.8/multiprocessing/queues.py", line 245, in _feed
    send_bytes(obj)
  File "/root/miniconda3/lib/python3.8/multiprocessing/connection.py", line 200, in send_bytes
    self._send_bytes(m[offset:offset + size])
  File "/root/miniconda3/lib/python3.8/multiprocessing/connection.py", line 411, in _send_bytes
    self._send(header + buf)
  File "/root/miniconda3/lib/python3.8/multiprocessing/connection.py", line 368, in _send
    n = write(self._handle, buf)
BrokenPipeError: [Errno 32] Broken pipe


New https://pypi.org/project/ultralytics/8.4.56 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.54 🚀 Python-3.8.10 torch-2.0.0+cu118 CUDA:0 (NVIDIA GeForce RTX 4080 SUPER, 32231MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=0, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_dataset/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.0, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.01, hsv_s=0.5, hsv_v=0.3, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yaml/baseline+M1+M2+M3(scdown).yaml, momentum=0.937, m

## 3.7 baseline+M1(36)

In [5]:
from ultralytics import YOLO
import torch

model = YOLO('yaml/baseline.yaml')

results = model.train(
    data='yolo_dataset/dataset.yaml',

    epochs=80,
    imgsz=1280,          # 从1280 → 960，大幅降低计算量（最有效）    
    
    batch=-1,           # 自动batch
    name='baseline+M1',
    device=0,

    optimizer='SGD',
    lr0=0.01,
    momentum=0.937,
    weight_decay=0.0005,

    warmup_epochs=3,
    warmup_momentum=0.8,

    mosaic=0.0,       
    copy_paste=0.0,
    mixup=0.0,

    hsv_h=0.010,
    hsv_s=0.5,
    hsv_v=0.3,

    translate=0.05,     # 降低平移强度
    scale=0.3,          # 降低缩放强度
    fliplr=0.5,
    erasing=0.0,        # 关闭随机擦除（减少计算）

    amp=True,           # 保留混合精度（必须开）
    cache='ram',
    workers=8,          # 从16→8，降低CPU负载（防止卡顿）
    
    patience=50,
    save_period=10,

    nbs=64,             # 累积batch，稳定训练
    close_mosaic=0,     # 保持关闭mosaic
    dropout=0.0,        # 关闭dropout（小模型不需要）
    rect=False,         # 关闭矩形训练（减少预处理）
)

WARNING ⚠️ no model scale passed. Assuming scale='n'.
New https://pypi.org/project/ultralytics/8.4.56 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.54 🚀 Python-3.8.10 torch-2.0.0+cu118 CUDA:0 (NVIDIA GeForce RTX 4080 SUPER, 32231MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=0, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_dataset/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.0, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.01, hsv_s=0.5, hsv_v=0.3, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=

## 3.8 baseline+M1(4)0.7

In [7]:
from ultralytics import YOLO
import torch

model = YOLO('yaml/baseline.yaml')

results = model.train(
    data='yolo_dataset/dataset.yaml',

    epochs=80,
    imgsz=1280,          # 从1280 → 960，大幅降低计算量（最有效）    
    
    batch=-1,           # 自动batch
    name='baseline+M1(4)0.7',
    device=0,

    optimizer='SGD',
    lr0=0.01,
    momentum=0.937,
    weight_decay=0.0005,

    warmup_epochs=3,
    warmup_momentum=0.8,

    mosaic=0.0,       
    copy_paste=0.0,
    mixup=0.0,

    hsv_h=0.010,
    hsv_s=0.5,
    hsv_v=0.3,

    translate=0.05,     # 降低平移强度
    scale=0.3,          # 降低缩放强度
    fliplr=0.5,
    erasing=0.0,        # 关闭随机擦除（减少计算）

    amp=True,           # 保留混合精度（必须开）
    cache='ram',
    workers=8,          # 从16→8，降低CPU负载（防止卡顿）
    
    patience=50,
    save_period=10,

    nbs=64,             # 累积batch，稳定训练
    close_mosaic=0,     # 保持关闭mosaic
    dropout=0.0,        # 关闭dropout（小模型不需要）
    rect=False,         # 关闭矩形训练（减少预处理）
)

WARNING ⚠️ no model scale passed. Assuming scale='n'.
New https://pypi.org/project/ultralytics/8.4.56 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.54 🚀 Python-3.8.10 torch-2.0.0+cu118 CUDA:0 (NVIDIA GeForce RTX 4080 SUPER, 32231MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=0, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_dataset/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.0, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.01, hsv_s=0.5, hsv_v=0.3, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=

## baseline+M1(64)

In [11]:
from ultralytics import YOLO
import torch

model = YOLO('yaml/baseline.yaml')

results = model.train(
    data='yolo_dataset/dataset.yaml',

    epochs=80,
    imgsz=1280,          # 从1280 → 960，大幅降低计算量（最有效）    
    
    batch=-1,           # 自动batch
    name='baseline+M1(64)',
    device=0,

    optimizer='SGD',
    lr0=0.01,
    momentum=0.937,
    weight_decay=0.0005,

    warmup_epochs=3,
    warmup_momentum=0.8,

    mosaic=0.0,       
    copy_paste=0.0,
    mixup=0.0,

    hsv_h=0.010,
    hsv_s=0.5,
    hsv_v=0.3,

    translate=0.05,     # 降低平移强度
    scale=0.3,          # 降低缩放强度
    fliplr=0.5,
    erasing=0.0,        # 关闭随机擦除（减少计算）

    amp=True,           # 保留混合精度（必须开）
    cache='ram',
    workers=8,          # 从16→8，降低CPU负载（防止卡顿）
    
    patience=50,
    save_period=10,

    nbs=64,             # 累积batch，稳定训练
    close_mosaic=0,     # 保持关闭mosaic
    dropout=0.0,        # 关闭dropout（小模型不需要）
    rect=False,         # 关闭矩形训练（减少预处理）
)

WARNING ⚠️ no model scale passed. Assuming scale='n'.
New https://pypi.org/project/ultralytics/8.4.56 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.54 🚀 Python-3.8.10 torch-2.0.0+cu118 CUDA:0 (NVIDIA GeForce RTX 4080 SUPER, 32231MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=0, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_dataset/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.0, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.01, hsv_s=0.5, hsv_v=0.3, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=

## 